In [20]:
import numpy as np
import pandas as pd
import pysam
import sys

In [21]:
BBfile = "minimap2_pRSVB_gRSVB.sam"
ABfile = "minimap2_pRSVA_gRSVB.sam"
outreport = "gap_amplicon_report.out"
outtable = "gap_amplicon_table.txt"
gapfile = "RSVB_gap_wide_regions.txt"


In [22]:
gaps = pd.read_csv(gapfile,sep="\t",index_col=False)
gaps

,gap,pname,pstart,pend,gstart,gend,size
0,gap1,RSVAB_7,1861,2363,1983,2206,223
1,gap2,RSVB_15-18,4325,5575,4459,5289,830
2,gap3,RSVB_30,8943,9353,9115,9214,99
3,gap4,RSVB_34,9974,10520,10185,10404,219
4,gap5,RSVB_44-45,13021,13736,13080,13627,547
5,gap6,RSVAB_48,14239,14648,14327,14559,232


In [23]:
def get_pools(samfile, start, end, pname='pool'):
    sam = pysam.AlignmentFile(samfile, "r")

    pool1 = list()
    pool2 = list()

    for x in sam:
        if x.is_unmapped: 
            #print(x.query_name,file=sys.stderr)
            continue
        if ((x.reference_start <= end) & (x.reference_end >=start)):
            if "_" in x.query_name:
                pindex = int(x.query_name.split("_")[1])
            pool = (pindex % 2)+1
            fr = ['F','R'][x.is_reverse]
            if(pool == 1):
                pool1.append((x.query_name, pname, pindex, pool, fr, x.reference_start, x.reference_end))
            elif(pool == 2):
                pool2.append((x.query_name, pname, pindex, pool, fr, x.reference_start, x.reference_end))
    return [pool1,pool2]

In [24]:
def get_amps(pool):
    amps = list()
    for p1 in pool:
        p1name, p1grp, p1pr, p1pool, p1fr, p1s, p1e = p1
        for p2 in pool:
            p2name, p2grp, p2pr, p2pool, p2fr, p2s, p2e = p2
            if ((p2s > p1s) & ((p1fr=='F') & (p2fr=='R'))):
                #print("{}:{}-->{}:{}\t{}\t{}".format(p1name,p1grp,p2name,p2grp, p1s, p2e))
                
                #amps.append((p1name,p1grp,p2name,p2grp, p1s, p2e))
                amps.append((p1s, p2e, p1name,p2name))
    return amps


In [25]:

gaplicons = list()

outfile = open(outreport,'w')

for i in range(0,len(gaps['gap'])):
    c = "NC_001781.1"
    gap = gaps.loc[i,'gap']
    pname = gaps.loc[i,'pname']
    s = gaps.loc[i,'pstart']
    e = gaps.loc[i,'pend']
    gs = gaps.loc[i,'gstart']
    ge = gaps.loc[i,'gend']

    ab1,ab2 = get_pools(ABfile,s,e,"A")
    bb1,bb2 = get_pools(BBfile,s,e,"B")
    
    pool1 = ab1+bb1
    pool2 = ab2+bb2
    amps1 = sorted(set(get_amps(pool1)))
    amps2 = sorted(set(get_amps(pool2)))
    
    
    print("==== {}  {}  {}-{} ====".format(gap,pname,gs,ge),file=outfile)
    print(" Pool 1 primers:",file=outfile)
    for p in pool1:
        print("  "," ".join(map(str,list(p))),file=outfile)
    print(" Pool 1 amplicons [{}]:".format(len(amps1)),file=outfile)
    for a in amps1:
        ast,aen,ap1,ap2 = a
        print("   {} -> {}\t{}-{}".format(ap1,ap2,ast,aen),file=outfile)
        gaplicons.append((gap,1,ap1,ap2,ast,aen))
    print("",file=outfile)
    print(" Pool 2 primers:",file=outfile)
    for p in pool2:
        print("  "," ".join(map(str,list(p))),file=outfile)
    print(" Pool 2 amplicons [{}]:".format(len(amps2)),file=outfile)
    for a in amps2:
        ast,aen,ap1,ap2 = a
        gaplicons.append((gap,2,ap1,ap2,ast,aen))
        print("   {} -> {}\t{}-{}".format(ap1,ap2,ast,aen),file=outfile)
    print("",file=outfile)
outfile.close()

In [26]:
gaplicontab = pd.DataFrame(gaplicons,columns=['gap','pool','p1','p2','start','end'])
gaplicontab.to_csv(outtable,sep="\t",index=False)
gaplicontab

,gap,pool,p1,p2,start,end
0,gap1,2,hRSVAB_7_LEFT_mod,RSVAB_7_RIGHT,1878,2381
1,gap2,1,RSVAB_16_LEFT,RSVB_16_RIGHT,4675,5051
2,gap2,1,RSVAB_16_LEFT,RSVA_16_RIGHT,4675,5440
3,gap2,1,RSVA_22_LEFT,RSVB_16_RIGHT,4690,5051
4,gap2,1,RSVA_22_LEFT,RSVA_16_RIGHT,4690,5440
5,gap2,1,RSVA_20_RIGHT,RSVB_16_RIGHT,4730,5051
6,gap2,1,RSVA_20_RIGHT,RSVA_16_RIGHT,4730,5440
7,gap2,1,RSVA_26_RIGHT,RSVB_16_RIGHT,4975,5051
8,gap2,1,RSVA_26_RIGHT,RSVA_16_RIGHT,4975,5440
9,gap2,1,RSVB_18_LEFT,RSVA_16_RIGHT,5303,5440
